[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyneuro/4590_Colabs/blob/main/10_ModelToBMI.ipynb)

# **Set 10 — BMI-1: micro:bit Only (LEGO–Colab)**
**Author:** Nair SS, et al., Electrical Engineering and Computer Science, University of Missouri-Columbia, Columbia, MO, USA

**Goal:** Provide a physical "FEAR" command bridge from a neural simulation to a micro:bit over USB UART. This module serves as a Brain-Machine Interface (BMI) capstone, translating digital states into physical expressions.

> **Note:** This notebook requires a **Local Runtime** for serial port access. See the Roadmap for instructions.

---

## **Table of Contents**
* [Variable Key and Roadmap](#variable-key-and-roadmap)
* [H0 — Starter: The Physical I/O Layer](#h0-starter-the-physical-io-layer)
* [H1 — FEAR Activation Logic (Host Side)](#h1-fear-activation-logic-host-side)
* [H2 — micro:bit Firmware: The Expression Layer](#h2-microbit-firmware-the-expression-layer)
* [Reflection: The BMI Synthesis](#reflection-the-bmi-synthesis)
* [Practice / Discussion Questions](#practice-discussion-questions)

---
### A Note on This Notebook
**This completed, filled-in notebook is the lab deliverable.** Add your own cells as you go. **H1 — FEAR Activation Logic** below is a good spot for an optional (AI-assisted) extension.

---
### 📋 Before You Begin: How This Module Is Graded
Within this module's combined Submissions + Quizzes score, the companion quiz is worth **66.7%** and the Colab submission is worth the remaining **33.3%**. Submissions + Quizzes together (across all 10 modules) make up **30%** of your overall course grade. The quiz specifically checks whether the understanding behind your submission is actually yours — pasting a classmate's code or an AI's explanation can get this notebook looking complete, but it won't get you through a quiz question applying the same concept to a scenario you haven't seen before.

The **design problem** below, and the **two questions you'll create yourself**, both exist for exactly this reason.

---

## **H0 — Starter: The Physical I/O Layer**
*Initializing the USB Bridge*

**Idea:** Before the brain can control the body, we must establish a nervous system. In this case, that system is a USB serial connection. This cell detects your micro:bit, opens a communication port, and defines the `send_fear()` function—the bridge that sends an ASCII command to the hardware.

#### **Predict → Verify**
* **Prediction:** Running this cell without the micro:bit plugged in (or while using a Cloud Runtime) will trigger a `RuntimeError`.
* **Verification:** Once plugged in and connected to a **Local Runtime**, the console should print `H0 ready.` This confirms the physical layer is active.

In [1]:
%pip  install ipywidgets
%pip -q install pyserial
%pip install jupyter_http_over_ws

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 3.0 MB/s eta 0:00:00


In [2]:
#Either this cell must be run locally, or the command must be used on your local CMD for the Colab connection to be stablished!
#The command commented out is the only thing needed to initialize a server. If you are able to run it in your CMD, it will display the server key for Colab.
#The code below them is mimicking your CMD in the notebook if you were to use the commands.

#!jupyter notebook --NotebookApp.allow_origin='https://colab.research.google.com' --port=8888 --NotebookApp.port_retries=0

#After the code/command has been run, copy the link created and paste it on Google Colab!
#In Google Colab go to the expandable menu on the upper right corner next to the RAM/Disk usage 'Additional connection options'.
#Once in 'Additional connection options' click on 'Connect to a Local Runtime' and input the link.

import subprocess
import sys

process = subprocess.Popen(
    [
        'jupyter', 'notebook',
        '--NotebookApp.allow_origin=https://colab.research.google.com',
        '--port=8888',
        '--NotebookApp.port_retries=0'
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

# Jupyter prints the token URL to stderr
for line in process.stderr:
    print(line, end='')
    sys.stdout.flush()

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
[I 20:48:21.437 NotebookApp] Writing notebook server cookie secret to /root/.local/share/jupyter/runtime/notebook_cookie_secret
[I 20:48:22.255 NotebookApp] Loading IPython parallel extension
[I 20:48:22.378 NotebookApp] [Jupytext Server Extension] Deriving a TextFileContentsManager from LargeFileManager
[C 20:48:23.639 NotebookApp] Running as root is not recommended. Use --allow-root to bypass.


In [3]:
#Port verifiaction to check that the microbit is detected

import serial.tools.list_ports as list_ports

for p in list_ports.comports():
    print(f"Device: {p.device}")
    print(f"  Description: {p.description}")
    print(f"  VID: {getattr(p, 'vid', None)}")
    print(f"  PID: {getattr(p, 'pid', None)}")
    print()

In [4]:
# H0 - The Physical I/O Layer


import time, sys, serial, serial.tools.list_ports as list_ports

# Standard identifiers for the BBC micro:bit
VID_MICROBIT = 3368
PID_MICROBIT = 516
BAUD = 115200
_ser = None

def find_port():
    """Detects the micro:bit COM port based on VID/PID or Description."""
    for p in list_ports.comports():
        if getattr(p, 'vid', None) == VID_MICROBIT and getattr(p, 'pid', None) == PID_MICROBIT:
            return p.device
    for p in list_ports.comports():
        d = (p.description or '').lower()
        if any(x in d for x in ['micro:bit', 'mbed', 'daplink', 'bbc']):
            return p.device
    return None

def open_serial():
    """Opens the serial port if it isn't already open."""
    global _ser
    if _ser and _ser.is_open:
        return _ser
    port = find_port()
    if not port:
        raise RuntimeError('micro:bit not found. Check USB connection and ensure you are on a Local Runtime.')
    _ser = serial.Serial(port, BAUD, timeout=0.2)
    time.sleep(0.2) # Allow for hardware handshake
    return _ser

def send_fear():
    """Writes the ASCII 'FEAR' command to the micro:bit."""
    s = open_serial()
    s.write(b'FEAR\n') # Newline helps the receiver parse the command
    s.flush()

print('H0 ready. Physical bridge established.')

H0 ready. Physical bridge established.


In [7]:
#Using the commands above to connect the server to the microbit via UART

find_port()
open_serial()

RuntimeError: micro:bit not found. Check USB connection and ensure you are on a Local Runtime.

## **H1 — FEAR Activation Logic (Host Side)**
*Defining the Decision Boundary*

**Idea:** Your neural model produces a fluid probability of fear (ranging from 0 to 1). In the physical world, a robot needs discrete actions. This section implements a **Threshold Rule** and a **Cooldown** (refractory period) to turn internal states into binary triggers.

#### **The Control Levers**
* **`thr` (Threshold):** How sensitive the robot is. A low threshold (0.3) makes the robot "jumpy"; a high threshold (0.8) makes it "brave."
* **`cd_ms` (Cooldown):** How long the robot must wait before it can be triggered again. This prevents "event storms" where the hardware tries to play multiple sounds at once.

#### **Exercises**
1.  **Manual Override:** Use the **"Test FEAR"** button to verify the hardware reacts instantly, regardless of the slider.
2.  **Sensitivity Sweep:** Lower the threshold to 0.1 and observe the print statements. Then raise it to 0.9.
3.  **Refractory Period:** Try clicking the test button very fast. Notice how the code ignores triggers that happen during the cooldown.

**Make It Your Own (optional AI-assisted extension).** The threshold logic above sends a binary FEAR/no-FEAR command. Extend it to send a **graded** signal instead (e.g., three levels: low/medium/high fear probability), and update the micro:bit firmware in H2 to respond differently to each level (e.g., dim vs. bright LED, or a different tone).

> **Suggested prompt:** *"Here is my FEAR threshold logic: [paste your H1 code]. I want to send 3 discrete levels instead of a binary signal over the same UART connection. Show me what to change here and in the micro:bit firmware below."*

*Your Answer Here (what you changed, and how you tested it):*

In [ ]:
# H1 - Activation Logic
import ipywidgets as w
from IPython.display import display

# UI Elements for interactive tuning
thr = w.FloatSlider(value=0.7, min=0, max=1, step=0.01, description='FEAR thr')
b = w.Button(description='Test FEAR', button_style='danger')

def on_fear(prob, threshold=0.7, cd_ms=1500):
    """
    Translates model probability into a hardware trigger.
    Includes a 'Refractory Period' (cooldown) to protect the hardware.
    """
    if not hasattr(on_fear, 't0'):
        on_fear.t0 = 0 # Initialize the timer

    now = time.time() * 1000 # Current time in milliseconds

    # Decision Logic: Must cross threshold AND be outside the cooldown window
    if prob >= threshold and (now - on_fear.t0) > cd_ms:
        send_fear()
        on_fear.t0 = now
        print(f'FEAR Command Sent! (Model State: {prob:.2f})')

def _manual_trigger_click(_):
    send_fear()
    print("Manual Trigger Sent via UI.")

# Connect the button and display the dashboard
b.on_click(_manual_trigger_click)
display(w.VBox([thr, b]))

## **H2 — micro:bit Firmware: The Expression Layer**
*Flash as main.py*

**Idea:** Now we move to the hardware side. This is a device-side event loop that listens for the "FEAR" command over UART (Universal Asynchronous Receiver-Transmitter). When it hears the signal, it executes a physical action—in this case, displaying a skull and playing a minor-key tone.

#### **How to Flash**
1.  Open the [Python Editor for micro:bit](https://python.microbit.org/).
2.  **Copy** the code cell below and **Paste** it into the web editor.
3.  Click **Connect** and then **Flash**.
4.  Once the yellow light on the back stops blinking, the micro:bit is ready. Return here to test it using the button in **H1**.

#### **Exercises**
1.  **Custom Expression:** Change `Image.SKULL` to `Image.GHOST` or `Image.SAD`.
2.  **Audio Profile:** The current tone is a descending "alarm." Can you change the `music.play` list to a high-pitched "startle" sound?

In [ ]:
# H2 - micro:bit Firmware (MicroPython)
from microbit import *
import music, uart

# Initialize UART to match the 115200 Baud rate in H0
uart.init(baudrate=115200)

def fear_response():
    """The physical expression of the fear state."""
    display.show(Image.SKULL)
    # Play a 'startle' sequence: notes and durations
    music.play(['c5:1', 'b4:1', 'a4:2'])
    display.clear()

# Main Event Loop
while True:
    if uart.any():
        # Read the incoming message and clean up formatting
        msg = uart.readline().strip().upper()

        # Check if the message matches our 'contract' from H0
        if msg == b'FEAR':
            fear_response()

    # Minimal sleep to prevent the CPU from overheating
    # while staying responsive to incoming commands
    sleep(10)

### 🎯 Design Problem — Apply Threshold Tuning
**Scenario:** Your FEAR activation logic (H1) uses a decision threshold on the incoming sensor signal to decide when to trigger the micro:bit's fear expression. The sensor line has some baseline noise.

**If the threshold is set too low, what's the likely failure mode? Too high?**

A) Too low → false triggers from background noise; Too high → real signals get missed entirely
B) Too low → real signals get missed; Too high → false triggers from noise
C) Threshold level has no effect on triggering behavior, only on timing
D) Both too-low and too-high thresholds produce identical behavior

*Reason it out before revealing the answer.*

**Answer: A).**

A threshold set too low means ordinary background noise can cross it, triggering false activations. A threshold set too high means the real signal may never reliably cross it, causing missed detections. This is the exact same trade-off from Set 4's BCI data bridge (`data_offset` tuning) — now applied to real physical hardware instead of a simulated signal.

**Note:** this module doesn't yet have a formal categorized Knowledge Bank the way earlier Sets do — that's still on the to-do list. This design problem and the create-your-own activity below are built directly from the existing H0–H2 content in the meantime.

### 📸 Upload Checkpoint 1 — The Threshold-Tuning Reasoning, From Memory
**Before you write anything: stop, close this notebook (or at minimum, don't scroll back up).** Without looking back, redo this reasoning from memory — same scenario (threshold too low vs. too high), same two failure modes — from scratch.

Only after you've written what you can recall should you check it against your own earlier work in this notebook. Then photograph your hand-written page and upload it below.

**Activity — vibe-code it:** ask an AI assistant to help you write a short Colab cell that lets you upload an image file (`from google.colab import files`) and display it inline.

In [ ]:
# Your image upload + display code here:
# Image upload + display
import io
from PIL import Image
from IPython.display import display

try:
    from google.colab import files
    uploaded = files.upload()
    for name, data in uploaded.items():
        print(name)
        display(Image.open(io.BytesIO(data)))

except ImportError:
    # Local runtime fallback
    import ipywidgets as w

    uploader = w.FileUpload(accept='image/*', multiple=False)
    out = w.Output()

    def _show(change):
        vals = uploader.value
        items = vals.values() if isinstance(vals, dict) else vals  # ipywidgets 7 vs 8
        with out:
            out.clear_output()
            for f in items:
                display(Image.open(io.BytesIO(bytes(f['content']))))

    uploader.observe(_show, names='value')
    display(uploader, out)


### 🎯 Your Turn — Design Two Questions, Not Just Answer Them
Pick any concept from this Set. Create **two** short questions of your own — one of each type:

**1. An Analysis-type question:** present a scenario, plot, or result, and ask the reader to interpret *why* something happened or what it implies.

**2. A Design-type question:** ask the reader *how* to achieve a specific target outcome, or what change would produce a given effect.

For each: include a brief setup, 2–4 possible answers, the correct answer, and a short explanation of why. No required format or topic beyond that — design both however makes sense to you. If your questions would genuinely catch someone who only skimmed this Set, you've demonstrated real understanding.

*Your Analysis-type question, answers, and explanation here:*

*Your Design-type question, answers, and explanation here:*

### 📸 Upload Checkpoint 2 — Sketch the Full Signal Pipeline
Without looking back through this notebook, sketch the complete pipeline from memory: the neural model's internal state, through the threshold/cooldown logic (H1), across the UART connection, to the micro:bit firmware (H2), to the physical output (LED/tone). Label each stage and what crosses between them.

**Activity — vibe-code it:** ask an AI assistant to help you write a short Colab cell that lets you upload an image file and display it inline.

In [ ]:
# Image upload + display
import io
from PIL import Image
from IPython.display import display

try:
    from google.colab import files
    uploaded = files.upload()
    for name, data in uploaded.items():
        print(name)
        display(Image.open(io.BytesIO(data)))

except ImportError:
    # Local runtime fallback
    import ipywidgets as w

    uploader = w.FileUpload(accept='image/*', multiple=False)
    out = w.Output()

    def _show(change):
        vals = uploader.value
        items = vals.values() if isinstance(vals, dict) else vals  # ipywidgets 7 vs 8
        with out:
            out.clear_output()
            for f in items:
                display(Image.open(io.BytesIO(bytes(f['content']))))

    uploader.observe(_show, names='value')
    display(uploader, out)


## **Practice / Discussion Questions — Set 10**
*Translation to Brain-Machine Interfaces*

1.  **System Design:** Define a specific *internal model state* you would export to a prosthetic device and justify why it is a meaningful control signal.
2.  **Thresholding:** Why is a simple threshold rule often preferred over complex ML pipelines for real-time educational BMIs?
3.  **Timing Mismatch:** *Predict:* What happens if the internal neural events occur at a much faster frequency than the physical effector (the robot) can react? How do you fix this mismatch?
4.  **Error Modes:** Identify one common *communication error mode* (e.g., buffer overflow, serial disconnect) and propose a software mitigation strategy.
5.  **Encoding Graded States:** If you wanted the robot to show "levels" of fear (e.g., dim LED for low fear, bright for high fear), how would you change the UART contract?
6.  **Debouncing:** Why is it important to "smooth" or debounce the neural signal before it reaches the threshold logic in H1?
7.  **Jitter Sources:** List two sources of *timing jitter* that could make the hardware response feel inconsistent even if the model is running perfectly.
8.  **Verification:** Design a simple "Loopback Test" to verify that the micro:bit is receiving exactly what the Colab is sending.
9.  **Scaling:** If you added a second effector (a motor for movement), how would you expand the "FEAR" command to handle multi-effector outputs?
10. **Mechanistic Connection:** How does Set 10 reinforce the idea that behavior is an emergent property of underlying mechanistic circuit logic?
